In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model = "gpt-4o-mini",temperature=0.2)

response = llm.invoke([HumanMessage(content="What is the capital of India?")])

In [ ]:
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model = "gpt-4o-mini",temperature=0.2)

@traceable(name ="plan-itenary",tags=["travel","planning"])
def plan_itenary(destination:str,days:int,budget:int):
    messages = [SystemMessage(content="You are a travel planner"),
                HumanMessage(content=f"Plan {days} trip to {destination} with {budget} USD budget")]
    response = llm.invoke(messages)
    return response.content

@traceable(name = "validate",run_type="tool")
def validate_budget(budget:int, days:int):
    daily = budget/days
    return {"daily_budget":daily,"feasible":daily>50}

## I am going to pass few variables -> budget and days and place


validation = validate_budget(1000,7)
if validation["feasible"]:
    print("Plan is feasible")
    plan = plan_itenary("Bali",7,1000)
    print(plan)

Plan is feasible
Planning a 7-day trip to Bali with a budget of $1,000 is definitely feasible, especially if you prioritize budget accommodations, local dining, and free or low-cost activities. Here’s a suggested itinerary along with tips to help you stay within your budget.

### Budget Breakdown
- **Accommodation**: $200 (average $30/night)
- **Food**: $140 (average $20/day)
- **Transportation**: $100 (scooter rental and local transport)
- **Activities**: $200 (tours, entrance fees, etc.)
- **Miscellaneous**: $60 (souvenirs, snacks, etc.)
- **Emergency Fund**: $300 (for unexpected expenses)

### Itinerary

#### Day 1: Arrival in Bali
- **Accommodation**: Check into a budget guesthouse or hostel in Kuta or Seminyak.
- **Activity**: Relax at Kuta Beach, explore the local area.
- **Food**: Try local warungs (small restaurants) for affordable meals.

#### Day 2: Ubud Exploration
- **Transportation**: Rent a scooter for the week (~$5/day).
- **Activity**: Visit the Ubud Monkey Forest ($5),

In [9]:
!pip3 install langfuse

  Using cached opentelemetry_exporter_otlp_proto_common-1.34.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_proto-1.34.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_sdk-1.34.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached protobuf-5.29.6-cp38-abi3-macosx_10_9_universal2.whl.metadata (592 bytes)
  Using cached opentelemetry_api-1.34.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_semantic_conventions-0.55b1-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 4.0 MB/s  0:00:00
Using cached opentelemetry_exporter_otlp_proto_common-1.34.1-py3-none-any.whl (18 kB)
Using cached opentelemetry_proto-1.34.1-py3-none-any.whl (55 kB)
Using cached opentelemetry_sdk-1.34.1-py3-none-any.whl (118 kB)
Using cached opentelemetry_api-1.34.1-py3-none-any.whl (65 kB)
Using cached opentelemetry_semantic_conventions-0.55b1-py3-none-any.whl (196 kB)
Using cached protobuf-5.29.6-cp38-abi3-macosx_10_9_univers

In [1]:
from langfuse import Langfuse
from openai import OpenAI

import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI

In [3]:
from langfuse import get_client

langfuse = get_client()
model = OpenAI()

with langfuse.start_as_current_observation(name = "travel-query",as_type="span") as span:
    span.update(input = "suggest a 5 day budget trip in India")

    with langfuse.start_as_current_observation(name = "destination-suggest",as_type="generation",model="gpt-4o-mini") as generation:
        response = model.chat.completions.create(
            model = "gpt-4o-mini",
            messages=[{"role":"user",
                       "content":"suggest a 5 day budget trip in India"}]
        )

        output = response.choices[0].message.content


        generation.update(output=output, usage_details={"input":response.usage.prompt_tokens,
                                                        "Output":response.usage.completion_tokens})
langfuse.flush()
print(output)


Certainly! Here’s a suggested 5-day budget trip itinerary for India, focusing on a mix of culture, nature, and local experiences. For this trip, we'll explore the vibrant city of Jaipur, known as the Pink City, along with a day trip to nearby Agra to see the Taj Mahal. 

### Day 1: Arrival in Jaipur
**Morning:**
- Arrive in Jaipur. 
- Check into a budget guesthouse or hostel (many budget options are available in the city starting from ₹500-₹1500 per night).
- Have breakfast at a local eatery (try traditional Rajasthani dishes).

**Afternoon:**
- Visit the Hawa Mahal (Palace of Winds) – entry fee is nominal.
- Explore Jantar Mantar, the astronomical observatory – entry fee around ₹50.

**Evening:**
- Stroll through the colorful Johari Bazaar for shopping (try some block-printed textiles, jewelry, and handicrafts).
- Dinner at a local restaurant (budget around ₹300-₹500).

### Day 2: Jaipur Sightseeing
**Morning:**
- Visit the Amer Fort. You can either walk up or take an elephant ride (e

In [4]:
## Langchain + Langfuse


import os

from dotenv import load_dotenv 

from langfuse.langchain import CallbackHandler

from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage,SystemMessage

from langchain_core.tools import tool

load_dotenv()

True

In [6]:
langfuse_handler = CallbackHandler()
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0.2)

In [ ]:
@tool
def validate_budget(budget:int, days:int):
    """
    Validate whether the budget is feasibile
    """
    daily = budget/days
    if daily > 50:
        return(f"Budget is feasibile , Daily Budget is ${daily:.2f}")
    else:
         return(f"Budget is very tight , Daily Budget is ${daily:.2f}")

In [8]:
def plan_itenary(destination:str,days:int,budget:int):

    budget_check = validate_budget.invoke({"budget":budget,"days":days})
    
    messages = [SystemMessage(content="You are a travel planner"),
                HumanMessage(content=f"Plan {days} trip to {destination} with total budget {budget} USD and budget validation {budget_check}. Include Itenary, hotels, transportation,food,misc. ")]
    response = llm.invoke(messages,config={"callbacks":[langfuse_handler]})
    return response.content

In [9]:
plan = plan_itenary("Bali",7,1000)
langfuse.flush()
print(plan)

Planning a 7-day trip to Bali with a total budget of $1,000 is feasible with careful management of expenses. Below is a detailed itinerary that includes accommodation, transportation, food, and miscellaneous expenses.

### **Trip Overview**
- **Destination:** Bali, Indonesia
- **Duration:** 7 Days
- **Total Budget:** $1,000
- **Daily Budget:** $142.86

### **Itinerary**

#### **Day 1: Arrival in Bali**
- **Accommodation:** Budget Hotel/Hostel in Kuta (e.g., The Island Hotel Bali) - $30/night
- **Transportation:** Airport transfer to hotel - $15
- **Food:** Local warung (restaurant) - $10
- **Miscellaneous:** SIM card for internet - $10
- **Total Day 1:** $65

#### **Day 2: Explore Kuta**
- **Accommodation:** Budget Hotel/Hostel in Kuta - $30
- **Transportation:** Rent a scooter for the day - $5
- **Food:** Breakfast at hotel, lunch at a local café, dinner at a beachside restaurant - $20
- **Activities:** Visit Kuta Beach, Waterbom Bali (entrance fee $35)
- **Total Day 2:** $125

#### *

In [1]:
!pip3 install prometheus_client


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
from prometheus_client import start_http_server,Counter,Histogram,Gauge

import time
from functools import wraps



LLM_REQUEST = Counter("llm_request_total","Total LLM Request", ["model","node"])

LLM_ERRORS = Counter("llm_error_total","Total LLM Error", ["model","node","error_type"])

LLM_LATENCY = Histogram("llm_latency_seconds","LLM Latency",["model","node"],buckets=(0.1,0.5,1,2,5,10))

TOKEN_USAGE = Counter("llm_token_usage","LLM Token Usage", ["model","token_type"])

ACTIVE_REQUEST = Gauge("llm_active_request","Current LLM Request")

ACTIVE_SESSIONS = Gauge("llm_active_session","Current active Users ")

In [2]:
def record_llm_metrics(model:str,node:str,latency:str,input_tokens:int =0, output_tokens:int = 0, error: str | None = None):
    
    LLM_REQUEST.labels(model = model, node=node).inc()

    LLM_LATENCY.labels(model = model,node=node).observe(latency)

    TOKEN_USAGE.labels(model=model,token_type="input").inc(input_tokens)

    TOKEN_USAGE.labels(model=model,token_type="output").inc(output_tokens)
    
    if error:
        LLM_ERRORS.labels(model=model,node=node,error_type = error).inc()




In [3]:
# =========================
# Decorator
# =========================

def monitor_llm(model: str, node: str):
    """
    Automatically tracks latency,
    success/failure and tokens.
    """

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            ACTIVE_REQUEST.inc()

            start_time = time.time()

            try:
                result = func(*args, **kwargs)

                latency = time.time() - start_time

                input_tokens = result.get("input_tokens", 0)
                output_tokens = result.get("output_tokens", 0)

                record_llm_metrics(
                    model=model,
                    node=node,
                    latency=latency,
                    input_tokens=input_tokens,
                    output_tokens=output_tokens
                )

                return result

            except Exception as e:

                latency = time.time() - start_time

                record_llm_metrics(
                    model=model,
                    node=node,
                    latency=latency,
                    error=type(e).__name__
                )

                raise

            finally:
                ACTIVE_REQUEST.dec()

        return wrapper

    return decorator




In [4]:
@monitor_llm(model ="gpt-4o-mini",node="planner")
def planner_node():
    time.sleep(1.5)

    return {
        "response":"Trip Planned",
        "input_token":100,
        "output_token":50
    }

@monitor_llm(model ="gpt-4o-mini",node="validation")
def validation_node():
    time.sleep(1.5)

    raise TimeoutError("LLM Timedout")

    

In [5]:
if __name__=="__main__":
    start_http_server(8000)

    print("Promethesus is running at http://localhost:8000")

    ACTIVE_SESSIONS.set(10)

    while True:
        try:
            planner_node()
        except Exception:
            pass

        try:
            validation_node()
        except Exception:
            pass

        time.sleep(2)

Promethesus is running at http://localhost:8000


KeyboardInterrupt: 